# NoisiQ — Week 4 Notebook
**Noise-Aware Quantum Circuit Simulation and Visualization**
*Period: April 13 – April 19, 2026 (overflow into Week 5)*

---

## Purpose of This Notebook

This notebook serves as the living record and demonstration for Week 4 of the NoisiQ project.

By the end of this notebook you should be able to:
- Apply Pauli noise channels (depolarizing, dephasing, bit-flip, phase-flip) to a circuit
- Run a many-shot simulation and collect aggregate error statistics via `ManyShotRunner`
- Display an error heatmap with halo-color effect overlaid on the circuit diagram
- Display a per-qubit error bar chart
- Confirm the Quirk-style theme constants are applied to all visualizations

---

## Week 4 Milestone Summary

The goal of Week 4 is to complete **Pauli noise channels**, a **many-shot runner**, the first
**aggregate error heatmap** (with error-rate halo as proxy for downstream impact), a
**per-qubit error bar chart**, and to establish the Quirk-inspired visual theme. Also includes
dependency updates: `bloqade-tsim` and `networkx` as core deps, `qiskit-aer` as optional.

---

## Status Tracker

| Task | Owner | Status |
|------|-------|--------|
| `pyproject.toml` — add `bloqade-tsim` + `networkx` as core deps, `[aer]` optional | DS | ⏳ In Progress |
| `noisiq/noise/pauli_channels.py` — depolarizing, dephasing, bit-flip, phase-flip | TJ | ✅ Done |
| `noisiq/backends/many_shot_runner.py` — N-shot runner + AggregateResult | TJ | ✅ Done |
| `noisiq/visualization/theme.py` — Quirk-style color constants | TJ | ✅ Done |
| `noisiq/visualization/charts/heatmap.py` — circuit print-out with error-rate halo | TJ | ✅ Done |
| `noisiq/visualization/charts/charts.py` — `plot_qubit_error_bar()`, `plot_fidelity_decay()` | TJ | ✅ Done |
| `tests/noise/test_paulichannel.py` | TJ | ✅ Done |
| `tests/backends/test_many_shot_runner.py` | TJ | ✅ Done |
| `tests/visualization/test_heatmap.py` | TJ | ✅ Done |
| `tests/visualization/test_theme.py` | TJ | ✅ Done |
| `tests/visualization/test_charts.py` | TJ | ✅ Done |
| All tests passing via `pytest` | TJ | ✅ Done |
| CI passing on GitHub | TJ | ⏳ In Progress |
| Week 4 demo section of this notebook complete | TJ | ✅ Done |

---

## File Build Order

```
1. pyproject.toml                                    ← Dependency updates
2. noisiq/noise/pauli_channels.py                   ← Pauli channel math  ✅
3. noisiq/backends/many_shot_runner.py              ← Runner + AggregateResult  ✅
4. noisiq/visualization/theme.py                    ← Color constants  ✅
5. noisiq/visualization/charts/heatmap.py           ← Heatmap with halo  ✅
6. noisiq/visualization/charts/charts.py            ← Bar chart + fidelity decay  ✅
7. tests/noise/test_paulichannel.py                 ← Tests  ✅
8. tests/backends/test_many_shot_runner.py          ← Tests  ✅
9. tests/visualization/test_heatmap.py              ← Tests  ✅
10. tests/visualization/test_theme.py               ← Tests  ✅
11. tests/visualization/test_charts.py              ← Tests  ✅
```

---

## Physics Requirements for Week 4

**Pauli channels** must satisfy:
- **Probability normalization:** `p_x + p_y + p_z ≤ 1` (remaining probability is identity/no-error)
- **Trace preservation:** guaranteed when probabilities sum ≤ 1
- **Complete positivity:** Pauli channels are automatically CP by construction

**Specific channels:**
- Depolarizing: `p_x = p_y = p_z = p/3` — equal chance of X, Y, or Z error
- Dephasing: `p_z = p` — Z errors only (phase randomization)
- Bit-flip: `p_x = p` — X errors only
- Phase-flip: `p_z = p` — physically identical to dephasing; different name for pedagogical clarity

**Heatmap halo (Week 4 proxy):** intensity = `counts_matrix.sum(axis=0) / max` normalized per gate.
Color scale: bright red (highest relative error count) → light blue (zero errors).
Upgraded to true downstream impact metric in Week 7.

---

## Notes and Decisions Log

| Date | Note | Name |
|------|------|------|
| 2026-04-13 | Week 4 started. Pauli channels and many-shot runner are the top priority. | DS |
| 2026-04-21 | `pauli_channels.py` complete. Four channel classes (DepolarizingChannel, DephaseChannel, BitFlipChannel, PhaseFlipChannel), each parameterized by a single `p` in [0, 1], all wrapping `PauliError` via `to_pauli_error()`. Abstract base `PauliChannel` enforces the interface. | DS |
| 2026-04-21 | `many_shot_runner.py` complete. `ManyShotRunner.run()` derives deterministic per-shot seeds from a top-level seed, calls `StimTableauBackend.run_single_shot()` N times, accumulates error events into `counts_matrix` (shape: n_qubits × n_ops). `AggregateResult` is a frozen dataclass exposing `error_rate_matrix`, `n_qubits`, `n_timesteps` as properties. | DS |
| 2026-04-21 | `theme.py` complete. Gate colors by category: Clifford (#5B9BD5 lite blue), T-gates (#2F75B6 traditional blue), other (#1F4E79 dark navy). Halo colormap: light blue (#A8D8F0) → bright red (#FF1744) via `LinearSegmentedColormap`. `gate_color()` and `halo_color()` are the primary public functions. `apply_global_style()` sets rcParams globally. | DS |
| 2026-04-21 | Moved heatmap and charts into `noisiq/visualization/charts/` subdirectory (rather than flat files) for better organization as the visualization module grows. `noisiq.visualization.charts` is now a subpackage. | DS |
| 2026-04-21 | `heatmap.py` complete. Circuit print-out uses `FancyBboxPatch` for rounded gate boxes colored by category. Halo drawn as a slightly larger `FancyBboxPatch` behind each gate, opacity 0.55, intensity normalized so the highest-error gate = 1.0. Includes colorbar. | DS |
| 2026-04-21 | `charts.py` complete. `plot_qubit_error_bar` sums `counts_matrix` along timestep axis and plots a horizontal bar per qubit. `plot_fidelity_decay` added now (uses Poisson approximation for zero-error fraction) — primary use is Week 7/9 suppression demos but available from Week 4. | DS |
| 2026-04-21 | Added `networkx` as core dependency — required for fan-out propagation graph (Week 7). | DS |
| 2026-04-21 | All Week 4 tests written and passing: `test_paulichannel.py` (13 tests), `test_many_shot_runner.py` (15 tests), `test_heatmap.py` (13 tests), `test_theme.py` (17 tests), `test_charts.py` (18 tests). | DS |
| | | |

# Installation Instructions (Developer)

`noisiq` not yet published on PyPI, install from source:

```bash
git clone https://github.com/dtsaxum-oss/noisiq.git
cd noisiq
pip install -e .
```

For optional Aer backend support:
```bash
pip install -e ".[aer]"
```

Verify:
```bash
python -c "import noisiq as nq; print(f'noisiq {nq.__version__} installed successfully!')"
```

In [ ]:
# ── Week 4 Demo Setup ──────────────────────────────────────────────────────
# Apply global style once so all figures in this notebook use the NoisiQ theme.

import numpy as np
import matplotlib.pyplot as plt

import noisiq as nq
from noisiq.visualization.theme import apply_global_style

apply_global_style()
print(f"noisiq {nq.__version__} ready")

---
## 1 — Pauli Noise Channels

Four named single-parameter channels, each wrapping `PauliError`. The table below
shows how each channel maps `p` onto the underlying X/Y/Z error probabilities.

In [ ]:
from noisiq.noise import (
    DepolarizingChannel,
    DephaseChannel,
    BitFlipChannel,
    PhaseFlipChannel,
)

p = 0.15
channels = [
    DepolarizingChannel(p=p),
    DephaseChannel(p=p),
    BitFlipChannel(p=p),
    PhaseFlipChannel(p=p),
]

print(f"{'Channel':<22} {'p_x':>6} {'p_y':>6} {'p_z':>6}  describe()")
print("-" * 72)
for ch in channels:
    err = ch.to_pauli_error()
    d   = ch.describe()
    print(f"{repr(ch):<22} {err.p_x:>6.3f} {err.p_y:>6.3f} {err.p_z:>6.3f}  {d['channel']}")

In [ ]:
# Sampling demo — BitFlipChannel(p=0.2) should produce ~20% X errors
rng = np.random.default_rng(42)
bit_flip = BitFlipChannel(p=0.2)
samples  = [bit_flip.to_pauli_error().sample(rng) for _ in range(2000)]

counts = {p: samples.count(p) for p in ("I", "X", "Y", "Z")}
print("BitFlipChannel(p=0.2) — 2000 samples:")
for pauli, n in counts.items():
    bar = "█" * (n // 20)
    print(f"  {pauli}: {n:>4}  ({n/20:.1f}%)  {bar}")

---
## 2 — Many-Shot Runner

`ManyShotRunner.run()` executes N independent shots of a noisy circuit and accumulates
error events into a `counts_matrix` of shape `(n_qubits, n_operations)`.
Each cell records how many times an error occurred on that qubit at that gate across all shots.

In [ ]:
from noisiq.ir import Circuit, gates
from noisiq.backends import ManyShotRunner

# 3-qubit GHZ circuit — H on q0, then CNOT chain
circuit = Circuit(n_qubits=3, name="ghz_noisy")
circuit.add_gate(gates.H,    (0,))
circuit.add_gate(gates.CNOT, (0, 1))
circuit.add_gate(gates.CNOT, (1, 2))

# 2% depolarizing noise on every gate
depol = DepolarizingChannel(p=0.02)
noise_config = {i: depol.to_pauli_error() for i in range(len(circuit.operations))}

N_SHOTS = 1000
runner = ManyShotRunner()
result = runner.run(circuit, n_shots=N_SHOTS, noise_config=noise_config, seed=42)

print(f"Circuit:          {circuit.n_qubits} qubits, {len(circuit.operations)} operations")
print(f"Shots:            {result.n_shots}")
print(f"counts_matrix shape:  {result.counts_matrix.shape}  (n_qubits × n_ops)\n")

print("counts_matrix (raw error events per cell):")
print(result.counts_matrix)

print("\nerror_rate_matrix (counts / n_shots):")
print(result.error_rate_matrix.round(4))

In [ ]:
# Reproducibility check — same seed must produce identical counts
result_a = runner.run(circuit, n_shots=200, noise_config=noise_config, seed=7)
result_b = runner.run(circuit, n_shots=200, noise_config=noise_config, seed=7)
result_c = runner.run(circuit, n_shots=200, noise_config=noise_config, seed=99)

print("Same seed → identical counts_matrix:", np.array_equal(result_a.counts_matrix, result_b.counts_matrix))
print("Diff seed → different counts_matrix:", not np.array_equal(result_a.counts_matrix, result_c.counts_matrix))

---
## 3 — Visual Theme

`theme.py` is the single source of truth for all colors and fonts. Any visualization
file imports from here rather than hardcoding values.

In [ ]:
from noisiq.visualization.theme import (
    gate_color, halo_color, get_halo_colormap,
    CLIFFORD_GATE_COLOR, T_GATE_COLOR, OTHER_GATE_COLOR,
    WIRE_COLOR, ERROR_COLOR,
)

# Gate color lookup
print("Gate color by category:")
for name in ("H", "CNOT", "CZ", "T", "T_DAG", "RZ", "U3"):
    print(f"  {name:<8} → {gate_color(name)}")

# Halo color at key intensities
print("\nHalo color by intensity:")
for intensity in (0.0, 0.25, 0.5, 0.75, 1.0):
    rgba = halo_color(intensity)
    print(f"  {intensity:.2f}  →  R={rgba[0]:.2f}  G={rgba[1]:.2f}  B={rgba[2]:.2f}")

# Visual swatch — gate category colors
fig, axes = plt.subplots(1, 3, figsize=(6, 1.2))
for ax, (label, color) in zip(axes, [
    ("Clifford", CLIFFORD_GATE_COLOR),
    ("T-gate",   T_GATE_COLOR),
    ("Other",    OTHER_GATE_COLOR),
]):
    ax.set_facecolor(color)
    ax.text(0.5, 0.5, label, color="white", ha="center", va="center",
            fontsize=11, fontweight="bold", transform=ax.transAxes)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("Gate category palette", fontsize=10, y=1.08)
plt.tight_layout()
plt.show()

# Halo colormap swatch
fig, ax = plt.subplots(figsize=(5, 0.6))
gradient = np.linspace(0, 1, 256).reshape(1, -1)
ax.imshow(gradient, aspect="auto", cmap=get_halo_colormap())
ax.set_xticks([0, 128, 255])
ax.set_xticklabels(["0.0\n(zero errors)", "0.5", "1.0\n(max impact)"], fontsize=8)
ax.set_yticks([])
ax.set_title("Halo colormap  (light blue → bright red)", fontsize=9)
plt.tight_layout()
plt.show()

---
## 4 — Error Heatmap

The heatmap renders a circuit "print-out" with a halo-color effect around each gate box.
Halo intensity is normalized across all gates: the gate with the highest total error count
gets the brightest red; zero errors gets light blue.

Three circuits are compared below at increasing noise levels to show the halo scaling.

In [ ]:
from noisiq.visualization.charts import plot_error_heatmap

# Compare three noise levels side-by-side using the GHZ circuit from Section 2
fig, axes = plt.subplots(1, 3, figsize=(14, 3))

for ax, (label, p_noise) in zip(axes, [
    ("Low noise  p=0.005",    0.005),
    ("Medium noise  p=0.05",  0.05),
    ("High noise  p=0.20",    0.20),
]):
    nc = {i: DepolarizingChannel(p=p_noise).to_pauli_error()
          for i in range(len(circuit.operations))}
    r = runner.run(circuit, n_shots=1000, noise_config=nc, seed=42)
    plot_error_heatmap(r, circuit, title=label, ax=ax)

plt.suptitle("GHZ circuit — halo intensity scales with error rate", fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Also show the heatmap for different channel types on the same circuit
# to confirm dephasing only lights up Z-prone paths vs depolarizing (all Paulis)
fig, axes = plt.subplots(1, 2, figsize=(10, 3))

for ax, (label, channel) in zip(axes, [
    ("Depolarizing  p=0.10",  DepolarizingChannel(p=0.10)),
    ("Dephasing  p=0.10",     DephaseChannel(p=0.10)),
]):
    nc = {i: channel.to_pauli_error() for i in range(len(circuit.operations))}
    r  = runner.run(circuit, n_shots=1000, noise_config=nc, seed=42)
    plot_error_heatmap(r, circuit, title=label, ax=ax)

plt.suptitle("Channel type comparison — same p, different error structure", fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

---
## 5 — Per-Qubit Error Bar Chart

`plot_qubit_error_bar` sums `counts_matrix` along the timestep axis to show which qubit
accumulated the most errors overall across the entire circuit.

In [ ]:
from noisiq.visualization.charts import plot_qubit_error_bar

# Use the main result from Section 2 (GHZ, 2% depolarizing, 1000 shots)
fig = plot_qubit_error_bar(result, title="GHZ circuit — per-qubit error rate (depolarizing p=0.02)")
plt.show()

# Print the underlying numbers for verification
print("Qubit-level summary:")
for q in range(result.n_qubits):
    total_errors = result.counts_matrix[q].sum()
    rate = result.error_rate_matrix[q].sum()
    print(f"  q{q}: {total_errors} total errors across {result.n_shots} shots  (rate {rate:.4f})")

---
## 6 — Putting It All Together

A 5-qubit circuit with mixed noise: depolarizing on single-qubit gates, dephasing on CNOTs.
Shows all four Week 4 outputs together in a single workflow.

In [ ]:
# Build a 5-qubit circuit: H layer (t=0), CNOT chain (t=1..4), X layer (t=5)
# CNOT(q, q+1) gates share qubit q+1 with the next CNOT, so they are physically
# sequential — each gets its own time step. X gates on distinct qubits can share t=5.
c5 = Circuit(n_qubits=5, name="five_qubit_mixed")
for q in range(5):
    c5.add_gate(gates.H, (q,), t=0)
for q in range(4):
    c5.add_gate(gates.CNOT, (q, q + 1), t=q + 1)   # t=1,2,3,4 — sequential chain
for q in range(5):
    c5.add_gate(gates.X, (q,), t=5)

# Mixed noise: depolarizing on H and X, dephasing on CNOTs
noise_map = {}
for i, op in enumerate(c5.operations):
    if op.gate.name.upper() in ("H", "X"):
        noise_map[i] = DepolarizingChannel(p=0.03).to_pauli_error()
    elif op.gate.name.upper() == "CNOT":
        noise_map[i] = DephaseChannel(p=0.05).to_pauli_error()

result5 = runner.run(c5, n_shots=2000, noise_config=noise_map, seed=42)

# — Heatmap
plot_error_heatmap(result5, c5, title="5-qubit mixed noise — error heatmap")
plt.show()

# — Per-qubit bar
plot_qubit_error_bar(result5, title="5-qubit mixed noise — per-qubit error rate")
plt.show()

print(f"\nTotal error events: {result5.counts_matrix.sum()} across {result5.n_shots} shots")
print(f"Mean error rate per (qubit, gate) cell: {result5.error_rate_matrix.mean():.4f}")